# StyleTTS 2 Demo (CZ)

In [ ]:
# Go to the root of the repository
%cd ..

### Randomness

In [ ]:
from Modules.pts import set_random_seed

set_random_seed(0)

### Imports and packages

In [ ]:
# load packages
import sys
import os
import os.path as osp
import logging
import time
import yaml
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
from munch import munchify

from models import load_ASR_models, load_F0_models, build_model
from Modules.diffusion.sampler import (ADPM2Sampler, DiffusionSampler,
                                       KarrasSchedule)
from Utils.PLBERT.util import load_plbert
from utils import recursive_munch
from text_utils import TextCleaner
from Modules.pts import PTS

from tpp_ttstool import TppTtstool

%matplotlib inline
import IPython.display as ipd

### Settings

In [ ]:
# Set username
USER = os.environ["USER"]

TPP_PATH = f"/storage/plzen4-ntis/home/{USER}/GIT_repos/TPP/src"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Set path to TPP
sys.path.insert(0, TPP_PATH)

# Define bin and data for phonemizer
TTSTOOL_BIN_PATH = "./Utils/tts_tool/tts_tool"
TTSTOOL_DATA_PATH = "./Utils/tts_tool/data/frontend_ph-redu.json"

# Set voice directory
VOICE_DIR = "Exps/NeuOl"
MODEL_NAME = "model4tts.pth"
MODEL_PATH = osp.join(VOICE_DIR, MODEL_NAME)
CONFIG_PATH = osp.join(VOICE_DIR, "config2.processed.yml")

In [ ]:
# Text must be a list.
def g2p(tpp, texts):
    if isinstance(texts, str):
        texts = [texts]
    assert isinstance(texts, (list, tuple)), "The input text must be a list!"
    
    ph_output = []
    for t in texts:
        # Prepare phonemizer
        tpp.ssml_parse(t)
        # Iterate over phonetic inputs
        ph_strings = [ps.strip() for ps in tpp.to_sentences_phon() if ps.strip()]
        ph_output.append(" ".join(ph_strings))
    return ph_output

### Set up TPP and PTS

In [ ]:
# Set up TPP
tpp = TppTtstool("cs-cz", tts_tool_bin=TTSTOOL_BIN_PATH, tts_tool_data=TTSTOOL_DATA_PATH)

# Set up phoneme-to-speech
pts = PTS(
    CONFIG_PATH,
    MODEL_PATH,
    t=1.0,
    diffusion_steps=5,
    embedding_scale=1.0,
    speech_rate=1.0,
    use_glob_noise=False,
    fix_noise_in_ph_string=False,
    log_level=logging.DEBUG,
)

In [ ]:
# synthesize a text
text = ["Šestašedesátiletý nadšený hráč stolního tenisu vlastní mnoho objektů v kraji.", "Je to prostě borec."]

### Synthesize speech

#### Basic synthesis (5 diffusion steps)

In [ ]:
start = time.time()

# Generate and concatenate wavs
wavs = pts(g2p(tpp, text))
wav = np.concatenate(wavs)

rtf = (time.time() - start) / (len(wav) / 24000)
print(f"RTF = {rtf:5f}")
display(ipd.Audio(wav, rate=24000))

#### With higher diffusion steps (more diverse)
Since the sampler is ancestral, the higher the steps, the more diverse the samples are, with the cost of slower synthesis speed.

In [ ]:
start = time.time()

# Set higher diffusion steps
pts.diffusion_steps = 10

# Generate and concatenate wavs
wavs = pts(g2p(tpp, text))
wav = np.concatenate(wavs)

rtf = (time.time() - start) / (len(wav) / 24000)
print(f"RTF = {rtf:5f}")
display(ipd.Audio(wav, rate=24000))

### Speech expressiveness
The following section recreates the samples shown in [Section 6](https://styletts2.github.io/#emo) of the demo page.

#### With embedding_scale=1
This is the classifier-free guidance scale. The higher the scale, the more conditional the style is to the input text and hence more emotional. 

In [ ]:
texts = {}
texts["Happy"] = "Já jsem tak šťastný, že tomu ani nemohu uvěřit."
texts["Sad"] = "Je mi to strašně líto, ale já jsem s tím nemohl nic dělat."
texts["Angry"] = "Tak to jdi do prdele! Tohle se nedělá."
texts["Surprised"] = "Tak tomu nemohu uvěřit. Opravdu se to stalo?"

# Set PTS
pts.diffusion_steps = 10
pts.embedding_scale = 1.0  # basic emotion

for k, v in texts.items():
    start = time.time()
    wavs = pts(g2p(tpp, v))
    wav = np.concatenate(wavs)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    print(f"{k}: ")
    display(ipd.Audio(wav, rate=24000, normalize=True))

#### With embedding_scale=2

In [ ]:
texts = {}
texts["Happy"] = "Já jsem tak šťastný, že tomu ani nemohu uvěřit."
texts["Sad"] = "Je mi to strašně líto, ale já jsem s tím nemohl nic dělat."
texts["Angry"] = "Tak to jdi do prdele! Tohle se nedělá."
texts["Surprised"] = "Tak tomu nemohu uvěřit. Opravdu se to stalo?"

# Set PTS
pts.diffusion_steps = 10
pts.embedding_scale = 1.5  # more emotional

for k, v in texts.items():
    start = time.time()
    wavs = pts(g2p(tpp, v))
    wav = np.concatenate(wavs)
    rtf = (time.time() - start) / (len(wav) / 24000)
    print(f"RTF = {rtf:5f}")
    print(f"{k}: ")
    display(ipd.Audio(wav, rate=24000, normalize=True))

### Long-form generation
This section includes basic implementation of Algorithm 1 in the paper for consistent longform audio generation. The example passage is taken from [Section 5](https://styletts2.github.io/#long) of the demo page. 

In [ ]:
long_text = """Šestašedesátiletý nadšený hráč stolního tenisu vlastní a spravuje většinu areálu mezi Budvarem a Ternem v krajském městě. Zbytek má jeho manželka. V rozhovoru vzpomíná na první roky v nové zemi, na boom zmíněných stánků u hranic i to, proč si vietnamská menšina u nás získala své místo."""

In [ ]:
start = time.time()

# Set higher diffusion steps
pts.diffusion_steps = 10
pts.embedding_scale = 1.0
pts.t = 0.7

# Generate and concatenate wavs
wavs = pts(g2p(tpp, long_text))
wav = np.concatenate(wavs)

rtf = (time.time() - start) / (len(wav) / 24000)
print(f"RTF = {rtf:5f}")
display(ipd.Audio(wav, rate=24000))